# 13 — Cryptographic Verification Specifies Trust

**Leading specification:** received bytes are accepted only when their digest matches the published identity.

Notebook 07 established that content addressing makes identity follow content:

\[
I = H(C)
\]

Notebook 13 adds the verification step:

\[
V=\bigl(
H(C_{\mathrm{received}})
=
H(C_{\mathrm{published}})
\bigr)
\]

A network can move bytes without being trusted. The receiver accepts the artifact only after local verification.

## 1. Context

A content identity can be published before retrieval. The received object must then be checked against that identity.

The engineering split is:

```text
published identity
→ received bytes
→ local digest
→ compare
→ accept / reject
```

Notebook 13 answers:

> How do I know I received the correct object?

It does not require trusting the server, mirror, peer, or transport path. It requires verifying the bytes.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import sys
import zipfile
from dataclasses import dataclass, asdict

import matplotlib.pyplot as plt
import pandas as pd

# Robust paths whether run from repo root or notebooks/.
CWD = Path.cwd()
if (CWD / "src").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "src").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = Path("..").resolve()

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "13_cryptographic_verification"
SITE = REPO_ROOT / "site"

FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
SITE.mkdir(parents=True, exist_ok=True)

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from open_model_distribution.hashing import content_id, sha256_file
except Exception:
    def sha256_file(path, chunk_size=1024 * 1024):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for chunk in iter(lambda: handle.read(chunk_size), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def content_id(path, algorithm="sha256"):
        if algorithm != "sha256":
            raise ValueError("Only sha256 is currently supported.")
        return f"sha256:{sha256_file(path)}"

def verify_file(path: Path, published_digest: str) -> bool:
    return sha256_file(path) == published_digest

print(f"Repo root: {REPO_ROOT}")
print(f"Results:   {RESULTS}")

## 2. Verification variables

The verification condition is local:

\[
V=\bigl(
H(C_{\mathrm{received}})
=
H(C_{\mathrm{published}})
\bigr)
\]

The receiver recomputes the digest from the received bytes and compares it with the published digest.

In [ ]:
@dataclass(frozen=True)
class VerificationVariables:
    published_identity: str
    received_content: str
    local_digest: str
    verification: str
    decision: str

variables = VerificationVariables(
    published_identity="Published digest or content ID expected by the receiver",
    received_content="Bytes obtained from a server, mirror, peer, cache, or archive",
    local_digest="Digest recomputed locally from received bytes",
    verification="Equality check between local digest and published digest",
    decision="PASS accepts bytes; FAIL rejects bytes",
)

pd.DataFrame([asdict(variables)]).T.rename(columns={0: "meaning"})

## 3. Publish an artifact identity

Start with a deterministic toy model artifact. The publisher computes and publishes its digest.

In [ ]:
artifact_dir = RESULTS / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)

published_path = artifact_dir / "published-model.bin"

payload = (
    b"open-model-distribution verification notebook\n"
    + b"toy weights block\n"
    + bytes(range(256)) * 256
)

published_path.write_bytes(payload)

published_digest = sha256_file(published_path)
published_cid = content_id(published_path)

published_manifest = {
    "schema": "open-model-distribution.verification-manifest.v0",
    "name": "toy-verified-model",
    "artifact": str(published_path.relative_to(REPO_ROOT)),
    "algorithm": "sha256",
    "digest": published_digest,
    "content_id": published_cid,
    "size_bytes": published_path.stat().st_size,
    "verification_rule": "sha256(received_bytes) == published_digest",
}

published_manifest_path = RESULTS / "13_manifest.json"
published_manifest_path.write_text(json.dumps(published_manifest, indent=2), encoding="utf-8")

published_manifest

## 4. Verify received bytes

A correct copy verifies. A changed copy fails.

\[
V\in\{\text{PASS},\text{FAIL}\}
\]

In [ ]:
received_dir = RESULTS / "received"
received_dir.mkdir(parents=True, exist_ok=True)

received_good = received_dir / "received-good.bin"
received_changed = received_dir / "received-changed.bin"

shutil.copyfile(published_path, received_good)

changed = bytearray(published_path.read_bytes())
changed[1024] = (changed[1024] + 1) % 256
received_changed.write_bytes(bytes(changed))

verification_rows = []
for label, path in [
    ("published source", published_path),
    ("received good copy", received_good),
    ("received changed copy", received_changed),
]:
    local_digest = sha256_file(path)
    passed = local_digest == published_digest
    verification_rows.append({
        "artifact": label,
        "path": str(path.relative_to(REPO_ROOT)),
        "size_bytes": path.stat().st_size,
        "published_digest": published_digest,
        "local_digest": local_digest,
        "verification": "PASS" if passed else "FAIL",
        "accepted": passed,
    })

verification_df = pd.DataFrame(verification_rows)
verification_csv = RESULTS / "13_verification.csv"
verification_df.to_csv(verification_csv, index=False)

verification_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.3))
ax.axis("off")

items = [
    ("Published\ndigest", 0.10, 0.55),
    ("Download\nbytes", 0.32, 0.55),
    ("Hash\nreceived", 0.54, 0.55),
    ("Compare", 0.74, 0.55),
    ("PASS / FAIL", 0.91, 0.55),
]

for label, x, y in items:
    ax.text(
        x, y, label,
        ha="center", va="center", fontsize=13,
        bbox=dict(boxstyle="round,pad=0.42", fc="white", ec="black", lw=1.5),
        transform=ax.transAxes,
    )

for (_, x1, y1), (_, x2, y2) in zip(items[:-1], items[1:]):
    ax.annotate(
        "", xy=(x2 - 0.075, y2), xytext=(x1 + 0.075, y1),
        arrowprops=dict(arrowstyle="->", lw=2),
        xycoords=ax.transAxes,
    )

ax.text(
    0.5, 0.17,
    "The receiver trusts the verification rule, not the transport path.",
    ha="center", va="center", fontsize=12,
    transform=ax.transAxes,
)

ax.set_title("Verification pipeline", fontsize=16, pad=18)
fig.tight_layout()

pipeline_fig = FIGURES / "13_verification_pipeline.png"
fig.savefig(pipeline_fig, dpi=180)
plt.show()

pipeline_fig

## 5. Tampering demonstration

One changed byte produces a different digest. Verification fails even when the file size and location look plausible.

In [ ]:
tamper_summary = pd.DataFrame([
    {
        "case": "original",
        "digest_prefix": sha256_file(published_path)[:16],
        "matches_published": verify_file(published_path, published_digest),
    },
    {
        "case": "changed one byte",
        "digest_prefix": sha256_file(received_changed)[:16],
        "matches_published": verify_file(received_changed, published_digest),
    },
])

tamper_csv = RESULTS / "13_tampering.csv"
tamper_summary.to_csv(tamper_csv, index=False)

tamper_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.axis("off")

left_digest = sha256_file(published_path)[:16] + "…"
right_digest = sha256_file(received_changed)[:16] + "…"

ax.text(0.25, 0.78, "Original", ha="center", fontsize=15, weight="bold", transform=ax.transAxes)
ax.text(0.75, 0.78, "Changed one byte", ha="center", fontsize=15, weight="bold", transform=ax.transAxes)

ax.text(0.25, 0.54, f"SHA-256\n{left_digest}", ha="center", va="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="black", lw=1.5), transform=ax.transAxes)
ax.text(0.75, 0.54, f"SHA-256\n{right_digest}", ha="center", va="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="black", lw=1.5), transform=ax.transAxes)

ax.text(0.25, 0.24, "✓ verifies", ha="center", fontsize=16, transform=ax.transAxes)
ax.text(0.75, 0.24, "✗ rejects", ha="center", fontsize=16, transform=ax.transAxes)

ax.annotate("byte changed", xy=(0.63, 0.54), xytext=(0.38, 0.54),
            arrowprops=dict(arrowstyle="->", lw=2), ha="center", va="center", xycoords=ax.transAxes)

ax.set_title("Verification catches changed content", fontsize=16, pad=16)
fig.tight_layout()

tampering_fig = FIGURES / "13_tampering.png"
fig.savefig(tampering_fig, dpi=180)
plt.show()

tampering_fig

## 6. Trust boundary

The network is an untrusted transport layer. Verification happens at the receiver.

```text
Publisher publishes digest.
Network transports bytes.
Receiver recomputes digest.
Receiver accepts only matching bytes.
```

In [ ]:
trust_boundary = pd.DataFrame([
    {
        "zone": "publisher",
        "role": "publishes digest and manifest",
        "trusted_for": "identity statement",
        "not_trusted_for": "receiver-side verification",
    },
    {
        "zone": "network",
        "role": "moves bytes",
        "trusted_for": "availability only",
        "not_trusted_for": "content integrity",
    },
    {
        "zone": "receiver",
        "role": "hashes received bytes and compares digest",
        "trusted_for": "local verification decision",
        "not_trusted_for": "original authorship",
    },
])

trust_boundary_csv = RESULTS / "13_trust_boundary.csv"
trust_boundary.to_csv(trust_boundary_csv, index=False)

trust_boundary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.axis("off")

zones = [
    ("Publisher\npublishes digest", 0.17),
    ("Network\nmoves bytes", 0.50),
    ("Receiver\nverifies locally", 0.83),
]

for label, x in zones:
    ax.text(x, 0.62, label, ha="center", va="center", fontsize=14,
            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="black", lw=1.5),
            transform=ax.transAxes)

ax.annotate("digest + bytes", xy=(0.39, 0.62), xytext=(0.27, 0.62),
            arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)
ax.annotate("received bytes", xy=(0.72, 0.62), xytext=(0.60, 0.62),
            arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

ax.plot([0.335, 0.335], [0.18, 0.86], linestyle="--", lw=1.3, transform=ax.transAxes)
ax.plot([0.665, 0.665], [0.18, 0.86], linestyle="--", lw=1.3, transform=ax.transAxes)

ax.text(0.50, 0.27, "transport can be untrusted", ha="center", fontsize=12, transform=ax.transAxes)
ax.text(0.83, 0.27, "accept only if\nH(received)=H(published)", ha="center", fontsize=12, transform=ax.transAxes)

ax.set_title("Trust boundary: verification moves to the receiver", fontsize=16, pad=16)
fig.tight_layout()

trust_boundary_fig = FIGURES / "13_trust_boundary.png"
fig.savefig(trust_boundary_fig, dpi=180)
plt.show()

trust_boundary_fig

## 7. Verification manifest

A verification manifest records the information required to decide whether received bytes should be accepted.

Notebook 13 keeps signatures as a future extension. The current manifest contains algorithm, digest, size, content ID, and verification rule.

In [ ]:
manifest_table = pd.DataFrame([
    {"field": "schema", "value": published_manifest["schema"], "purpose": "manifest interpretation"},
    {"field": "algorithm", "value": published_manifest["algorithm"], "purpose": "hash function"},
    {"field": "digest", "value": published_manifest["digest"][:24] + "…", "purpose": "published identity"},
    {"field": "content_id", "value": published_manifest["content_id"][:32] + "…", "purpose": "addressable identity"},
    {"field": "size_bytes", "value": published_manifest["size_bytes"], "purpose": "basic consistency check"},
    {"field": "verification_rule", "value": published_manifest["verification_rule"], "purpose": "acceptance condition"},
])

manifest_table_csv = RESULTS / "13_manifest_table.csv"
manifest_table.to_csv(manifest_table_csv, index=False)

manifest_table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.axis("off")

ax.text(0.5, 0.92, "Verification manifest", ha="center", va="center", fontsize=16, weight="bold", transform=ax.transAxes)

fields = [
    ("algorithm", "sha256"),
    ("digest", published_digest[:18] + "…"),
    ("size", str(published_path.stat().st_size) + " bytes"),
    ("content_id", published_cid[:26] + "…"),
    ("rule", "H(received)=H(published)"),
]

y = 0.75
for field, value in fields:
    ax.text(0.28, y, field, ha="right", va="center", fontsize=12, weight="bold", transform=ax.transAxes)
    ax.text(0.32, y, value, ha="left", va="center", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", lw=1.1),
            transform=ax.transAxes)
    y -= 0.12

ax.annotate("", xy=(0.5, 0.10), xytext=(0.5, 0.25),
            arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)
ax.text(0.5, 0.06, "local verification decision", ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
manifest_fig = FIGURES / "13_manifest.png"
fig.savefig(manifest_fig, dpi=180)
plt.show()

manifest_fig

## 8. Verification report

The verification report summarizes what was accepted and what was rejected.

In [ ]:
verification_report = {
    "notebook": "13_cryptographic_verification.ipynb",
    "title": "Cryptographic Verification Specifies Trust",
    "published_manifest": str(published_manifest_path.relative_to(REPO_ROOT)),
    "published_digest": published_digest,
    "published_content_id": published_cid,
    "accepted_count": int(verification_df["accepted"].sum()),
    "rejected_count": int((~verification_df["accepted"]).sum()),
    "statement": "Received bytes are accepted only when their digest matches the published identity.",
    "records": verification_df.to_dict(orient="records"),
}

verification_report_path = RESULTS / "13_verification_report.json"
verification_report_path.write_text(json.dumps(verification_report, indent=2), encoding="utf-8")

verification_report

## 9. Engineering statement

Cryptographic verification specifies trust by moving the acceptance decision to the receiver. The network may provide bytes, but the receiver accepts only bytes whose digest matches the published identity. Trust follows local verification, not transport.

## 10. Key equations

Content identity:

\[
I = H(C)
\]

Verification condition:

\[
V=\bigl(
H(C_{\mathrm{received}})
=
H(C_{\mathrm{published}})
\bigr)
\]

Verification decision:

\[
V\in\{\text{PASS},\text{FAIL}\}
\]

Trust specification:

\[
\text{Trust}
=
\text{published identity}
+
\text{local verification}.
\]

## 11. Notebook outputs

Notebook 13 writes:

- `results/13_cryptographic_verification/13_manifest.json`
- `results/13_cryptographic_verification/13_verification.csv`
- `results/13_cryptographic_verification/13_tampering.csv`
- `results/13_cryptographic_verification/13_trust_boundary.csv`
- `results/13_cryptographic_verification/13_manifest_table.csv`
- `results/13_cryptographic_verification/13_verification_report.json`
- `figures/13_verification_pipeline.png`
- `figures/13_tampering.png`
- `figures/13_trust_boundary.png`
- `figures/13_manifest.png`

In [ ]:
notebook_manifest = {
    "notebook": "13_cryptographic_verification.ipynb",
    "title": "Cryptographic Verification Specifies Trust",
    "primary_specification": "cryptographic verification",
    "statement": "Received bytes are accepted only when their digest matches the published identity.",
    "equations": [
        "I = H(C)",
        "V = (H(C_received)=H(C_published))",
        "V in {PASS, FAIL}",
        "Trust = published identity + local verification",
    ],
    "outputs": {
        "manifest": str(published_manifest_path.relative_to(REPO_ROOT)),
        "verification_csv": str(verification_csv.relative_to(REPO_ROOT)),
        "tampering_csv": str(tamper_csv.relative_to(REPO_ROOT)),
        "trust_boundary_csv": str(trust_boundary_csv.relative_to(REPO_ROOT)),
        "manifest_table_csv": str(manifest_table_csv.relative_to(REPO_ROOT)),
        "verification_report": str(verification_report_path.relative_to(REPO_ROOT)),
        "verification_pipeline_figure": str(pipeline_fig.relative_to(REPO_ROOT)),
        "tampering_figure": str(tampering_fig.relative_to(REPO_ROOT)),
        "trust_boundary_figure": str(trust_boundary_fig.relative_to(REPO_ROOT)),
        "manifest_figure": str(manifest_fig.relative_to(REPO_ROOT)),
    },
}

notebook_manifest_path = RESULTS / "13_notebook_manifest.json"
notebook_manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")

notebook_manifest

## 12. Download artifacts

Run the final cell to package notebook 13 outputs for download.

In [ ]:
# Package notebook artifacts for download

from pathlib import Path
from IPython.display import FileLink, display
import zipfile

NOTEBOOKS = REPO_ROOT / "notebooks"
SITE = REPO_ROOT / "site"

zip_path = RESULTS / "13_cryptographic_verification_artifacts.zip"

notebook_path = NOTEBOOKS / "13_cryptographic_verification.ipynb"
fallback_notebook_path = Path.cwd() / "13_cryptographic_verification.ipynb"

paths_to_include = [
    notebook_path if notebook_path.exists() else fallback_notebook_path,
    FIGURES,
    RESULTS,
    SITE,
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)

        if item.is_dir():
            for path in item.rglob("*"):
                if path.is_file() and path.resolve() != zip_path.resolve():
                    try:
                        archive_name = path.relative_to(REPO_ROOT)
                    except ValueError:
                        archive_name = path.name
                    zf.write(path, archive_name)

        elif item.exists() and item.resolve() != zip_path.resolve():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")

display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass